In [0]:
%sql
-- MAGIC # Build Constructor Standings
-- MAGIC
-- MAGIC #### Sources
-- MAGIC 1. fact_session_results
-- MAGIC 1. dim_constructors
-- MAGIC
-- MAGIC #### Output Columns
-- MAGIC 1. season
-- MAGIC 1. constructor id
-- MAGIC 1. constructor name
-- MAGIC 1. nationality
-- MAGIC 1. race starts
-- MAGIC 1. total points
-- MAGIC 1. number of wins
-- MAGIC 1. number of podiums
-- MAGIC 1. standing position


In [0]:
%sql
CREATE OR REPLACE VIEW formula1_catalog.gold.v_constructor_standings 
AS
with constructor_summary as
(
select 
    f.season,
    d.constructor_id,
    d.constructor_name,
    d.nationality,
    count(*) as race_starts,
    sum(f.points) as total_points,
    count_if(f.is_win) as number_of_wins,
    count_if(f.is_podium) as number_of_podiums
from 
    formula1_catalog.gold.fact_session_results f
join 
    formula1_catalog.gold.dim_constructors d
on 
    f.constructor_id = d.constructor_id
group by
    f.season,
    d.constructor_id,
    d.constructor_name,
    d.nationality
)
select 
    season,
    constructor_id,
    constructor_name,
    nationality,
    rank() over (partition by season order by total_points desc,number_of_wins desc) as standing_position,
    race_starts,
    total_points,
    number_of_wins,
    number_of_podiums
from 
    constructor_summary;

select * from formula1_catalog.gold.v_constructor_standings where season=2025;